In [1]:
# Libraries
import requests
from datetime import date
import json
import pandas as pd
import plotly.express as px

### EXTRACT & LOAD API DATA

In [2]:
# Fetch API Data
# Create functions so it can easily reusable
def fetch_eq_api(params):
    """Fetch the earthquake data from USGS API"""
    url = "https://earthquake.usgs.gov/fdsnws/event/1/query"
    r = requests.get(url, params=params)

    # Check if the api call worked (200 = Success)
    if r.status_code == 200:
        print(f"Fetch success: {r.status_code}")
        return r.json()
    else:
        print(f"Fetch FAILED: {r.status_code}")
        return None

In [3]:
# the minmagnitude = 5.0 cause from USGS classified the start of moderate earthquake, less than that is light and minor 
params = {
    'format': 'geojson',
    'starttime': '2026-01-01',
    'endtime': date.today().isoformat(),
    'minmagnitude': 5.0
}

response_dicts = fetch_eq_api(params=params)
print(response_dicts.keys())
print(response_dicts['metadata'])

Fetch success: 200
dict_keys(['type', 'metadata', 'features', 'bbox'])
{'generated': 1791320243000, 'url': 'https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime=2026-01-01&endtime=2026-10-07&minmagnitude=5.0', 'title': 'USGS Earthquakes', 'status': 200, 'api': '2.7.0', 'count': 1532}


In [4]:
# create readable data and inspect what's in it
preview_dicts = response_dicts.copy()

if 'features' in preview_dicts:
    preview_dicts['features'] = preview_dicts['features'][:1]
    
print(json.dumps(preview_dicts, indent=4))

{
    "type": "FeatureCollection",
    "metadata": {
        "generated": 1791320243000,
        "url": "https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime=2026-01-01&endtime=2026-10-07&minmagnitude=5.0",
        "title": "USGS Earthquakes",
        "status": 200,
        "api": "2.7.0",
        "count": 1532
    },
    "features": [
        {
            "type": "Feature",
            "properties": {
                "mag": 5.5,
                "place": "92 km NNW of Aleneva, Alaska",
                "time": 1791311671760,
                "updated": 1791319991507,
                "tz": null,
                "url": "https://earthquake.usgs.gov/earthquakes/eventpage/aka2026tuxgky",
                "detail": "https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=aka2026tuxgky&format=geojson",
                "felt": 320,
                "cdi": 4.6,
                "mmi": 4.215,
                "alert": "green",
                "status": "reviewed",
              

In [5]:
# found out the data we need is in features, then pass it to variable response_dict
response_dict = response_dicts['features']
print(len(response_dict))

1532


In [6]:
# cause the data is nested, we can't just pass it to the pandas dataframe
# Flatten the nested JSON into a clean list of dictionaries
clean_data = []

for eq in response_dict:
    clean_data.append({
        "magnitude": eq["properties"]["mag"],
        "place": eq["properties"]["place"],
        "time": eq["properties"]["time"],
        "longitude": eq["geometry"]["coordinates"][0],
        "latitude": eq["geometry"]["coordinates"][1],
        "depth": eq["geometry"]["coordinates"][2]
    })

print(len(clean_data))

1532


In [7]:
# Turn it into a Pandas DataFrame
df = pd.DataFrame(clean_data)

# inspect
print(df.head())
print(df.info())
print(df.describe())

   magnitude                                           place           time  \
0        5.5                    92 km NNW of Aleneva, Alaska  1791311671760   
1        5.1                   34 km NNW of Bāgeshwar, India  1791305591296   
2        5.9           264 km SSW of Severo-Kuril’sk, Russia  1791305301805   
3        5.0                       south of the Fiji Islands  1791231779243   
4        5.0  187 km ENE of Saipan, Northern Mariana Islands  1791197529844   

   longitude  latitude    depth  
0  -153.6090   58.8020   87.600  
1    79.6148   30.1171   10.000  
2   154.8085   48.4554   26.047  
3   179.8108  -24.4235  518.638  
4   147.3950   15.8046   10.000  
<class 'pandas.DataFrame'>
RangeIndex: 1532 entries, 0 to 1531
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   magnitude  1532 non-null   float64
 1   place      1532 non-null   str    
 2   time       1532 non-null   int64  
 3   longitude  1532 non-n

In [8]:
# Convert the date datatype to datetime
df['time'] = pd.to_datetime(df['time'], unit='ms')

# Create a df copy for analysis
df_clean = df.copy()

# inspect
print(df_clean.head())
print(df_clean.info())
print(df_clean.describe())

   magnitude                                           place  \
0        5.5                    92 km NNW of Aleneva, Alaska   
1        5.1                   34 km NNW of Bāgeshwar, India   
2        5.9           264 km SSW of Severo-Kuril’sk, Russia   
3        5.0                       south of the Fiji Islands   
4        5.0  187 km ENE of Saipan, Northern Mariana Islands   

                     time  longitude  latitude    depth  
0 2026-10-06 18:34:31.760  -153.6090   58.8020   87.600  
1 2026-10-06 16:53:11.296    79.6148   30.1171   10.000  
2 2026-10-06 16:48:21.805   154.8085   48.4554   26.047  
3 2026-10-05 20:22:59.243   179.8108  -24.4235  518.638  
4 2026-10-05 10:52:09.844   147.3950   15.8046   10.000  
<class 'pandas.DataFrame'>
RangeIndex: 1532 entries, 0 to 1531
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   magnitude  1532 non-null   float64       
 1   place      1532 non-null  

### VISUALIZATION

#### 01. WORLD MAP EARTHQUAKE DISTRIBUTION

In [ ]:
# pip install nbformat to show the visual in jupyter
# dataframe = df_clean
df_clean['date_clean'] = df_clean['time'].dt.strftime('%Y-%m-%d')

title = "Global Distribution of Earthquakes (>= 5.0 Magnitude, 2026 YTD)"

fig = px.scatter_geo(
    data_frame=df_clean,
    lat='latitude',
    lon='longitude',
    title=title,
    size='magnitude',
    color='magnitude',
    color_continuous_scale='viridis',
    projection='natural earth',
    hover_name='place',
    hover_data={
        'magnitude': ': .1f',
        'date_clean': True,
        'latitude':': .2f',
        'longitude':': .2f'
    },
    labels={
        'date_clean': 'Date',
        'magnitude': 'Magnitude',
        'latitude': 'Latitude',
        'longitude': 'Longitude'
    },
    size_max=6,
    opacity=0.5,
)

fig.update_traces(marker_line_width=0)

# fig.write_html("charts/01_eq_global_distribution.html")
# fig.write_image("charts/01_eq_global_distribution.png", width=1200, height=700, scale=2)
fig.show()

#### 02. INDONESIA EARTHQUAKE DISTRIBUTION

In [ ]:
# df_clean['date_clean'] = df_clean['time'].dt.strftime('%Y-%m-%d')

country_name = 'Indonesia'
df_indonesia = df_clean[df_clean['place'].str.contains(country_name, case=False, na=False)]

title = "Indonesia Distribution of Earthquakes (>= 5.0 Magnitude, 2026 YTD)"

fig = px.scatter_geo(
    data_frame=df_indonesia,
    lat='latitude',
    lon='longitude',
    title=title,
    size='magnitude',
    color='magnitude',
    color_continuous_scale='viridis',
    projection='natural earth',
    hover_name='place',
    hover_data={
        'magnitude': ': .1f',
        'date_clean': True,
        'latitude':': .2f',
        'longitude':': .2f'
    },
    labels={
        'date_clean': 'Date',
        'magnitude': 'Magnitude',
        'latitude': 'Latitude',
        'longitude': 'Longitude'
    },
    size_max=8,
)

fig.update_traces(marker_line_width=0)

# fig.write_html("charts/02_eq_Indonesia.html")
# fig.write_image("charts/02_eq_Indonesia.png", width=1200, height=700, scale=2)
fig.show()

#### 03. INDONESIA EARTHQUAKE OVER TIME

In [ ]:
# df_clean['date_clean'] = df_clean['time'].dt.strftime('%Y-%m-%d')
# country_name = 'Indonesia'
# df_indonesia = df_clean[df_clean['place'].str.contains(country_name, case=False, na=False)]

df_indonesia['place_clean'] = df_indonesia['place'].str.replace(', Indonesia', '', regex=False)
df_timeline = df_indonesia.sort_values('date_clean', ascending=True)

title = "Indonesia's Earthquakes Over Time (>= 5.0 Magnitude, 2026 YTD)"

fig = px.scatter(
    data_frame=df_timeline,
    x='date_clean',
    y='magnitude',
    title=title,
    size='magnitude',           
    color='magnitude',
    color_continuous_scale=['#d0dff9', '#1361e1', '#da2129'],
    hover_name='place',
    size_max=12,
    hover_data={
        'magnitude': ': .1f',
        'date_clean': '|%Y-%m-%d',
        'latitude':': .2f',
        'longitude':': .2f'
    },
    labels={
        'date_clean': 'Date',
        'magnitude': 'Magnitude',
        'latitude': 'Latitude',
        'longitude': 'Longitude'
    },
)

fig.update_xaxes(
    dtick="M1",         
    tickformat="%b",    
    ticklabelmode="instant"
)
fig.update_layout(xaxis_title=None, yaxis_title=None)
fig.update_traces(marker_line_width=0)

# fig.write_html("charts/03_eq_ina_overtime.html")
# fig.write_image("charts/03_eq_ina_overtime.png", width=1200, height=700, scale=2)
fig.show()

#### 04. MONTHLY TRENDS

In [13]:
# df_clean['date_clean'] = df_clean['time'].dt.strftime('%Y-%m-%d')
# country_name = 'Indonesia'
# df_indonesia = df_clean[df_clean['place'].str.contains(country_name, case=False, na=False)]
# df_indonesia['place_clean'] = df_indonesia['place'].str.replace(', Indonesia', '', regex=False)
# df_timeline = df_indonesia.sort_values('date_clean', ascending=True)

df_timeline['month'] = pd.to_datetime(df_timeline['date_clean']).dt.to_period('M').astype(str)
monthly_trend = df_timeline.groupby(by='month').size().reset_index(name='count')

title = "Monthly Trend of Indonesia Earthquake (>= 5.0 Magnitude, 2026 YTD)"

fig = px.line(
    data_frame=monthly_trend,
    x='month',
    y='count',
    title=title,
    markers=True,
    hover_data={'count': True, 'month': True},
    labels={'count': 'Frequency'},
)

fig.update_xaxes(
    dtick="M1",         
    tickformat="%b",    
    ticklabelmode="instant"
)
fig.update_layout(xaxis_title=None, yaxis_title='Frequency')
fig.update_traces(
    line=dict(color='#1361e1', width=3),  
    marker=dict(size=8),                 
)

# fig.write_html("charts/04_eq_ina_monthlytrends.html")
# fig.write_image("charts/04_eq_ina_monthlytrends.png", width=1200, height=700, scale=2)
fig.show()

#### 05. TOP 10 INDONESIA'S BIGGEST EARTHQUAKE

In [14]:
# 5. Top 10 Biggest Eq 2026 ytd
# df_clean['date_clean'] = df_clean['time'].dt.strftime('%Y-%m-%d')
# country_name = 'Indonesia'
# df_indonesia = df_clean[df_clean['place'].str.contains(country_name, case=False, na=False)]
# df_indonesia['place_clean'] = df_indonesia['place'].str.replace(', Indonesia', '', regex=False)

df_top10 = df_indonesia.sort_values(by='magnitude', ascending=False).head(10)

title = "Indonesia's Top 10 Highest Magnitude Earthquake (>= 5.0 Magnitude, 2026 YTD)"

fig = px.bar(
    data_frame=df_top10,
    x='magnitude',
    y='place_clean',
    title=title,
    orientation='h',
    text='magnitude',
    color='magnitude',
    color_continuous_scale=['#d0dff9', '#1361e1'],
    hover_name='place',
    hover_data={
        'magnitude': ': .1f',
        'date_clean': '|%Y-%m-%d',
        'latitude':': .2f',
        'longitude':': .2f',
        'place_clean': False
    },
    labels={
        'date_clean': 'Date',
        'magnitude': 'Magnitude',
        'latitude': 'Latitude',
        'longitude': 'Longitude'
    },
)

fig.update_layout(xaxis_title='Magnitude', yaxis_title=None, coloraxis_showscale=False)
fig.update_yaxes(autorange='reversed')

# fig.write_html("charts/05_eq_ina_top10.html")
# fig.write_image("charts/05_eq_ina_top10.png", width=1200, height=700, scale=2)
fig.show()